# Model Training: Resume Job Category Classifier

This notebook trains the Logistic Regression job-category classifier used by the Streamlit app.

**Prerequisite:** Download `Resume.csv` from Kaggle and place it at `../data/Resume.csv` before running this notebook.

Pipeline:
1. Load and clean `Resume_str`
2. Encode `Category` labels
3. Train/test split
4. TF-IDF vectorization
5. Train Logistic Regression
6. Evaluate: accuracy, precision, recall, F1, confusion matrix
7. Save `model`, `vectorizer`, and `label_encoder` to `../models/`

In [ ]:
import sys
import os

# Add project root to path so `src` package imports work from within notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import clean_series
from src.predictor import train_and_save_model, DEFAULT_DATA_PATH, METRICS_PATH

print("Expected dataset path:", DEFAULT_DATA_PATH)

## 1. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DEFAULT_DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df["Category"].value_counts()

## 2. Preview text cleaning

In [ ]:
sample_raw = df["Resume_str"].iloc[0]
sample_clean = clean_series(df["Resume_str"].iloc[:1]).iloc[0]

print("RAW (first 400 chars):\n", sample_raw[:400])
print("\nCLEANED (first 400 chars):\n", sample_clean[:400])

## 3. Train the model

This calls the exact same `train_and_save_model` function used by `python -m src.predictor --train`,
so there is a single, real implementation of the training pipeline — no duplicated logic.

In [ ]:
metrics = train_and_save_model(data_path=DEFAULT_DATA_PATH, test_size=0.2, random_state=42, max_features=5000)

print("Accuracy:", metrics["accuracy"])
print("Precision (weighted):", metrics["precision_weighted"])
print("Recall (weighted):", metrics["recall_weighted"])
print("F1 (weighted):", metrics["f1_weighted"])
print("Train samples:", metrics["n_train_samples"], "| Test samples:", metrics["n_test_samples"])
print("Number of classes:", metrics["n_classes"])

## 4. Classification report

In [ ]:
report_df = pd.DataFrame(metrics["classification_report"]).transpose()
report_df

## 5. Confusion matrix heatmap

In [ ]:
import numpy as np

conf_matrix = np.array(metrics["confusion_matrix"])
labels = metrics["labels"]

plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix, xticklabels=labels, yticklabels=labels, cmap="Blues", annot=False)
plt.xlabel("Predicted Category")
plt.ylabel("True Category")
plt.title("Confusion Matrix - Job Category Classifier")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Confirm saved artifacts

In [ ]:
from src.predictor import MODEL_PATH, VECTORIZER_PATH, LABEL_ENCODER_PATH

for path in [MODEL_PATH, VECTORIZER_PATH, LABEL_ENCODER_PATH, METRICS_PATH]:
    print(path, "->", "EXISTS" if os.path.exists(path) else "MISSING")

Artifacts are now saved to `../models/`. You can launch the Streamlit app with `streamlit run app.py` from the project root.